# 第10章 移動体チャネルとドップラー

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第10章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# 本書の多くの章で使うミュンヘン中心部の都市シーン
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

# 受信機に速度を与えるとパスごとのドップラーが計算される
scene.add(Receiver('rx0', position=[60.0, -40.0, 1.5],
                   velocity=[10.0, 0.0, 0.0]))
paths = solver(scene, max_depth=5)
a, tau = paths.cir(normalize_delays=True, out_type='numpy')

## 2. 移動する物体

**注記**: 物体名はシーンごとに違う。`scene.objects` で実際の名前を確かめて置き換える。

In [ ]:
car = scene.objects['car-001']   # 物体名はシーンごとに違う
car.velocity = [10.0, 0.0, 0.0]  # +x 方向に 10 m/s

## 3. 速度ベクトルの設定

In [ ]:
rx = Receiver('rx0', position=[100.0, 50.0, 1.5],
              velocity=[10.0, 0.0, 0.0])  # x方向に 10 m/s
scene.add(rx)

## 4. パスごとのドップラーシフト

In [ ]:
# Receiver に velocity を設定して計算した各パスのドップラー
fD = np.array(paths.doppler).flatten()
valid = np.array(paths.valid).flatten()
fD_valid = fD[valid]
print('Doppler shift range:', fD_valid.min(), 'to',
      fD_valid.max(), 'Hz')

## 5. 時変 CIR を作る

In [ ]:
# 0.1 秒間、1 ms サンプルで時変 CIR を生成
t_axis = np.linspace(0, 0.1, 100)
H_t = []
for t in t_axis:
    # 各パスの位相を v . k_hat * t だけ回転
    a_t = a * np.exp(1j*2*np.pi*fD*t)
    H_t.append(a_t)
H_t = np.array(H_t)  # (num_time, num_paths)

## 7. 車両移動のシミュレーション

In [ ]:
v_kmh = 36; v = v_kmh/3.6  # m/s
trajectory = lambda t: [v*t - 200, 50, 1.5]
cir_seq = []
for t in np.linspace(0, 30, 151):  # 30 秒間を 0.2 秒刻み
    scene.remove('rx0')
    scene.add(Receiver('rx0', position=trajectory(t)))
    paths = solver(scene, max_depth=3)
    cir_seq.append(paths)